In [ ]:
! chmod 600 /content/kaggle.json
! KAGGLE_CONFIG_DIR=/content/ kaggle datasets download -d moltean/fruits

In [ ]:
import zipfile
zip_file = zipfile.ZipFile('/content/fruits.zip')
zip_file.extractall('/tmp/')

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [ ]:
train_generator = ImageDataGenerator(rescale=1/255,
                                     horizontal_flip=True,
                                     vertical_flip=True,
                                     rotation_range=60,
                                     zoom_range=0.3,
                                     fill_mode='nearest')

validation_generator = ImageDataGenerator(rescale=1/255,
                                     horizontal_flip=True,
                                     vertical_flip=True,
                                     rotation_range=60,
                                     zoom_range=0.3,
                                     fill_mode='nearest')

In [ ]:
train_data = train_generator.flow_from_directory('/tmp/fruits-360_dataset/fruits-360/Training',
                                                 batch_size=100,
                                                 class_mode='categorical',
                                                 target_size=(100, 100))

val_data = validation_generator.flow_from_directory('/tmp/fruits-360_dataset/fruits-360/Test',
                                                 batch_size=100,
                                                 class_mode='categorical',
                                                 target_size=(100, 100))

In [ ]:
def create_model():
  
  model = Sequential([   
      Conv2D(16, (3, 3), activation='relu', input_shape=(100, 100, 3)),
      MaxPooling2D(2, 2),
      Conv2D(32, (3, 3), activation='relu'),
      MaxPooling2D(3, 3),
      Flatten(), 
      Dense(128, activation='relu'), 
      Dense(131, activation='softmax')  
  ])
  
  model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy']) 
  
  return model

In [ ]:
class myCallback(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs={}):
            if logs.get('accuracy') > 0.99:
                print("\nReached 99% accuracy so cancelling training!") 
                self.model.stop_training = True

In [ ]:
model = create_model()

history = model.fit(train_data,
          epochs=100,
          verbose=1,
          validation_data=val_data)

In [ ]:
import matplotlib.pyplot as plt

acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']

epochs=range(len(acc)) 

plt.plot(epochs, acc, 'r', "Training Accuracy")
plt.plot(epochs, val_acc, 'b', "Validation Accuracy")
plt.title('Training and validation accuracy')
plt.show()
print("")

plt.plot(epochs, loss, 'r', "Training Loss")
plt.plot(epochs, val_loss, 'b', "Validation Loss")
plt.title('Training and validation loss')
plt.show()

In [ ]:
def download_history():
  import pickle
  from google.colab import files

  with open('history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

  files.download('history.pkl')

download_history()

In [ ]:
!mkdir -p saved_model
model.save('saved_model/my_model')

In [ ]:
!ls saved_model
!ls saved_model/my_model

In [ ]:
check_model = tf.keras.models.load_model('saved_model/my_model')
check_model.summary()